# FlowGuard — Full Pipeline
End-to-end: Setup → Preprocessing → MLP Baseline → Contrastive Pre-training → Federated Learning → Few-Shot Adaptation → Hardening → Evaluation

**Run cells top-to-bottom.** Each phase builds on the previous checkpoints.

---
## 0 · Setup & Validate

In [ ]:
import os
import shutil

# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 2. Clone / update the repo ────────────────────────────────────────────────
REPO_DIR = "/content/flowguard"
GITHUB_URL = "https://github.com/alperengoncu/guardian-engine-d3"

if not os.path.exists(REPO_DIR):
    !git clone "{GITHUB_URL}" "{REPO_DIR}"
else:
    !git -C "{REPO_DIR}" pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── 3. Copy datasets from Drive → data/raw/ ───────────────────────────────────
DRIVE_DATA_DIR = "/content/drive/MyDrive/flowguard"
RAW_DIR = os.path.join(REPO_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

EXPECTED_FILES = [
    "NF-UNSW-NB15-v3.csv",
    "NF-BoT-IoT-v3.csv",
    "NF-ToN-IoT-v3.csv",
    "NF-CICIDS2018-v3.csv",
]

for fname in EXPECTED_FILES:
    src = os.path.join(DRIVE_DATA_DIR, fname)
    dst = os.path.join(RAW_DIR, fname)
    if os.path.exists(dst):
        print(f"  [skip]  {fname} — already in data/raw/")
    elif os.path.exists(src):
        print(f"  [copy]  {fname} ({os.path.getsize(src)/1e9:.2f} GB) …", end=" ", flush=True)
        shutil.copy2(src, dst)
        print("done")
    else:
        print(f"  [MISSING] {fname} — not found in {DRIVE_DATA_DIR}")

print("\nData copy complete.")

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q flwr[simulation] shap umap-learn advertorch wandb pyarrow pyyaml tqdm plotly
!pip install -e . -q

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — go to Runtime > Change runtime type > GPU")

from src.utils.config import load_config, get_device
from src.utils.reproducibility import setup_reproducibility
from src.data.validate_raw import validate_raw_datasets

config = load_config("configs/base.yaml")
setup_reproducibility(config)
device = get_device(config)
print(f"Device: {device}")

report = validate_raw_datasets(config_path="configs/base.yaml")
all_ok = all(info["exists"] for info in report.values())
if not all_ok:
    print("\nMISSING DATASETS:")
    for ds_name, info in report.items():
        if not info["exists"]:
            print(f"  -> {info['expected_file']}")
    print("\nDownload from: https://staff.itee.uq.edu.au/marius/NIDS_datasets/")
else:
    print("\nAll datasets found!")
    for ds_name, info in report.items():
        print(f"  {ds_name}: {info['rows']:,} rows, {info['cols']} cols, {info['size_mb']:.1f} MB")

---
## 1 · EDA & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sample = pd.read_csv("data/raw/NF-UNSW-NB15-v3.csv", nrows=50000)
print(f"Shape: {sample.shape}")
print(f"\nLabel distribution:\n{sample['Label'].value_counts()}")
print(f"\nAttack types:\n{sample['Attack'].value_counts()}")

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, col in zip(axes.flat, ['IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS', 'PROTOCOL']):
    if col in sample.columns:
        sample[col].hist(ax=ax, bins=50)
        ax.set_title(col)
plt.tight_layout()
os.makedirs("results/plots", exist_ok=True)
plt.savefig("results/plots/raw_distributions.png", dpi=100)
plt.show()

In [ ]:
from src.data.preprocess import run_full_preprocessing

all_stats = run_full_preprocessing("configs/base.yaml")
print(f"\nProcessed {len(all_stats)} datasets.")
for name, stats in all_stats.items():
    print(f"  {name}: {len(stats.feature_names)} features")

In [ ]:
from src.data.splits import generate_all_splits
generate_all_splits("configs/base.yaml")

for name in ['unsw', 'bot', 'ton', 'cic']:
    path = f"data/processed/{name}.parquet"
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"{name}: {len(df):,} rows — label dist: {df['Label'].value_counts().to_dict()}")

---
## 2 · Phase 1: MLP Baseline

In [ ]:
from src.models.mlp_baseline import MLPBaseline
from src.data.dataset import create_dataloader
from src.data.features import get_input_dim
from src.training.supervised import SupervisedTrainer
from src.utils.config import load_config, get_device

config1 = load_config("configs/phase1_baseline.yaml")

results_a = {}
for ds_info in config1['data']['datasets']:
    name = ds_info['name']
    train_path = f"data/splits/protocol_a/{name}_train.parquet"
    val_path   = f"data/splits/protocol_a/{name}_val.parquet"
    if not os.path.exists(train_path):
        print(f"Skipping {name}")
        continue
    print(f"\n{'='*50}\nTraining MLP on: {name}\n{'='*50}")
    train_loader = create_dataloader(train_path, batch_size=1024, balanced=True)
    val_loader   = create_dataloader(val_path,   batch_size=1024, shuffle=False)
    model = MLPBaseline(input_dim=get_input_dim(), hidden_dims=[256, 128, 64], num_classes=2, dropout=0.3)
    trainer = SupervisedTrainer(model, train_loader, val_loader, config1,
                                checkpoint_dir=f"checkpoints/phase1/{name}/")
    results_a[name] = trainer.train()

In [ ]:
from src.evaluation.protocols import evaluate_protocol_a, evaluate_protocol_b

for ds_info in config1['data']['datasets']:
    name = ds_info['name']
    ckpt_path = f"checkpoints/phase1/{name}/best_model.pt"
    if not os.path.exists(ckpt_path):
        continue
    model = MLPBaseline(input_dim=get_input_dim(), hidden_dims=[256, 128, 64], num_classes=2)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device)
    print(f"\n--- {name} ---")
    evaluate_protocol_a(model, config1, device)

---
## 3 · Phase 2: Contrastive Pre-training

In [ ]:
assert os.path.exists("data/processed/combined_unlabeled.parquet"), "Run preprocessing first!"
print(f"Unlabeled data: {os.path.getsize('data/processed/combined_unlabeled.parquet') / 1e6:.0f} MB")

In [ ]:
from src.training.contrastive import train_contrastive
history = train_contrastive("configs/phase2_contrastive.yaml")

plt.figure(figsize=(10, 5))
plt.plot(history['loss'])
plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title('Contrastive Pre-training Loss')
plt.savefig('results/plots/phase2_loss.png', dpi=100)
plt.show()

In [ ]:
from src.models.transformer_encoder import FlowTransformerEncoder
from src.data.dataset import create_dataloader
from src.evaluation.visualization import extract_embeddings, plot_umap
from src.utils.config import load_config
import numpy as np

config2 = load_config("configs/phase2_contrastive.yaml")
enc_cfg = config2['model']['encoder']
encoder = FlowTransformerEncoder(
    input_dim=enc_cfg.get('input_dim', 49),
    model_dim=enc_cfg.get('model_dim', 128),
).to(device)
encoder.load_state_dict(torch.load("checkpoints/phase2/best_encoder.pt", map_location=device))

all_emb, all_labels, all_ds = [], [], []
for i, ds_info in enumerate(config2['data']['datasets']):
    name = ds_info['name']
    path = f"data/splits/protocol_a/{name}_test.parquet"
    if not os.path.exists(path):
        continue
    loader = create_dataloader(path, batch_size=1024, shuffle=False)
    emb, labels = extract_embeddings(encoder, loader, device, max_samples=2000)
    all_emb.append(emb); all_labels.append(labels); all_ds.extend([name] * len(emb))

all_emb = np.concatenate(all_emb)
all_labels = np.concatenate(all_labels)
plot_umap(all_emb, all_labels, label_names={0: 'Benign', 1: 'Attack'},
          title="Phase 2 Encoder Embeddings", save_path="results/plots/phase2_umap.png")

---
## 4 · Phase 3: Federated Learning

In [ ]:
from src.training.federated import run_federated_simulation
from src.utils.config import load_config

config3 = load_config("configs/phase3_federated.yaml")
history = run_federated_simulation("configs/phase3_federated.yaml")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, metrics in history['client_metrics'].items():
    axes[0].plot([m['loss'] for m in metrics], label=name)
    axes[1].plot([m['accuracy'] for m in metrics], label=name)
axes[0].set_title('Client Loss per Round'); axes[0].legend()
axes[1].set_title('Client Accuracy per Round'); axes[1].legend()
plt.savefig('results/plots/phase3_convergence.png', dpi=100)
plt.show()

In [ ]:
from src.models.flowguard import FlowGuard
from src.evaluation.protocols import evaluate_protocol_b

model = FlowGuard(config3)
model.load_state_dict(torch.load("checkpoints/phase3/final_global.pt", map_location=device))
model.to(device)
print("Protocol B — Federated Model:")
evaluate_protocol_b(model, config3, device)

---
## 5 · Phase 4: Few-Shot Adaptation

In [ ]:
from src.training.fewshot import run_fewshot_evaluation

results = run_fewshot_evaluation("configs/phase4_fewshot.yaml")

fig, ax = plt.subplots(figsize=(10, 6))
shots = [5, 10, 20, 50]
for ds_name, ds_results in results.items():
    f1s = [ds_results.get(s, {}).get('f1_macro', 0) for s in shots]
    ax.plot(shots, f1s, 'o-', label=ds_name)
ax.set_xlabel('Number of Shots'); ax.set_ylabel('F1 Macro')
ax.set_title('Few-Shot Adaptation Performance')
ax.legend(); ax.set_xticks(shots)
plt.savefig('results/plots/phase4_fewshot.png', dpi=100)
plt.show()

---
## 6 · Phase 5: Hardening (Domain Adversarial + EWC + PGD)

In [ ]:
from src.training.adversarial import domain_adversarial_training_step
from torch.cuda.amp import GradScaler
from src.utils.config import load_config

config5 = load_config("configs/phase5_hardening.yaml")

model = FlowGuard(config5)
ckpt_path = "checkpoints/phase3/final_global.pt"
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.enable_domain_discriminator(num_domains=4)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = torch.nn.CrossEntropyLoss()
scaler    = GradScaler(enabled=device.type == 'cuda')

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss, n = 0, 0
    for ds_idx, ds_info in enumerate(config5['data']['datasets']):
        name = ds_info['name']
        path = f"data/splits/protocol_a/{name}_train.parquet"
        if not os.path.exists(path):
            continue
        loader = create_dataloader(path, batch_size=512, num_workers=0)
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            domain_labels = torch.full((x.size(0),), ds_idx, dtype=torch.long, device=device)
            lambda_grl    = min(1.0, epoch / num_epochs)
            loss, _       = domain_adversarial_training_step(
                model, x, y, domain_labels, optimizer, criterion, lambda_grl)
            total_loss += loss * x.size(0); n += x.size(0)
    print(f"Epoch {epoch+1}/{num_epochs}: loss={total_loss/max(n,1):.4f}")

os.makedirs("checkpoints/phase5", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/phase5/hardened_model.pt")
print("Hardened model saved.")

In [ ]:
from src.training.ewc import compute_and_save_fisher

train_path = "data/splits/protocol_a/unsw_train.parquet"
if os.path.exists(train_path):
    loader = create_dataloader(train_path, batch_size=512, shuffle=False, num_workers=0)
    compute_and_save_fisher(model, loader, "checkpoints/phase5/fisher.pt", device=device)
    print("Fisher matrix computed and saved.")

---
## 7 · Full Evaluation

In [ ]:
import json
from src.evaluation.cross_dataset import run_full_evaluation
from src.evaluation.protocols import generate_comparison_table, ProtocolResult

all_results = {}

# Phase 1 — MLP Baseline
for ds_info in config['data']['datasets']:
    name = ds_info['name']
    ckpt = f"checkpoints/phase1/{name}/best_model.pt"
    if os.path.exists(ckpt):
        m = MLPBaseline(input_dim=get_input_dim(), num_classes=2)
        m.load_state_dict(torch.load(ckpt, map_location=device)['model_state_dict'])
        m.to(device)
        all_results[f"phase1_{name}"] = run_full_evaluation(m, config, phase_name=f"phase1_{name}")
        break

# Phase 3 — Federated
if os.path.exists("checkpoints/phase3/final_global.pt"):
    m = FlowGuard(config3)
    m.load_state_dict(torch.load("checkpoints/phase3/final_global.pt", map_location=device))
    m.to(device)
    all_results["phase3"] = run_full_evaluation(m, config3, phase_name="phase3_federated")

# Phase 5 — Hardened
if os.path.exists("checkpoints/phase5/hardened_model.pt"):
    m = FlowGuard(config5)
    m.load_state_dict(torch.load("checkpoints/phase5/hardened_model.pt", map_location=device))
    m.to(device)
    all_results["phase5"] = run_full_evaluation(m, config5, phase_name="phase5_hardened")

In [ ]:
flat_results = {}
for phase, phase_results in all_results.items():
    flat_results[phase] = []
    for protocol_key, result_list in phase_results.items():
        if isinstance(result_list, list):
            flat_results[phase].extend(result_list)

table = generate_comparison_table(flat_results)
print(table)

os.makedirs("results/metrics", exist_ok=True)
with open("results/metrics/comparison_table.md", "w") as f:
    f.write(table)

In [ ]:
from src.evaluation.explainability import compute_shap_values, plot_shap_summary, get_feature_importance
from src.data.features import get_feature_names

test_path = "data/splits/protocol_a/unsw_test.parquet"
if os.path.exists(test_path) and 'm' in dir():
    loader = create_dataloader(test_path, batch_size=256, shuffle=False, num_workers=0)
    print("Computing SHAP values (this may take a while)...")
    result = compute_shap_values(m, loader, get_feature_names(), device, num_samples=200)
    if result:
        shap_values, data, names = result
        os.makedirs("results/shap", exist_ok=True)
        plot_shap_summary(shap_values, data, names, save_path="results/shap/summary.png")
        importance = get_feature_importance(shap_values, names)
        print("\nTop 10 Most Important Features:")
        for feat, imp in importance[:10]:
            print(f"  {feat}: {imp:.4f}")